# 04 — Feature Extraction (UPGRADED)
Extracts all original features PLUS:
- **CDPP** — photometric noise level from FITS header
- **ACF period** — detects stellar rotation (filters false positives)
- **Lomb-Scargle period** — sinusoidal variability check
- **TLS features** — SDE, FAP, odd-even mismatch from notebook 03

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares, LombScargle
from astropy.stats import mad_std
import astropy.units as u
import lightkurve as lk
from tqdm import tqdm
import os, glob, warnings
warnings.filterwarnings('ignore')
print('Imports OK!')

Imports OK!


c:\Users\Alivia Hossain\Desktop\exoplanet_detection\Exoplanet_detection\.venv\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


## 2. Configuration

In [2]:
PROCESSED_DIR = '../data/processed/'
RAW_DIR       = '../data/raw/'
BLS_RESULTS   = '../outputs/bls_all_results.csv'
OUTPUT_FILE   = '../outputs/features.csv'
LABELED_FEAT  = '../outputs/labeled_features.csv'

bls_df = pd.read_csv(BLS_RESULTS)
bls_df['tic_id'] = bls_df['tic_id'].astype(str).str.replace('.0','', regex=False)
print(f'Loaded {len(bls_df)} BLS/TLS results')
print(f'Columns: {bls_df.columns.tolist()}')

Loaded 15 BLS/TLS results
Columns: ['tic_id', 'period_days', 'duration_days', 'depth_ppm', 'snr', 'bls_power', 'transit_time', 'tls_period', 'tls_duration', 'tls_depth_ppm', 'tls_snr', 'tls_sde', 'tls_fap', 'tls_rp_rs', 'tls_odd_even', 'tls_transit_time', 'bls_tls_period_agree_pct', 'both_agree']


## 3. NEW — CDPP Extraction from FITS Header

In [3]:
def get_cdpp(tic_id, raw_dir):
    """
    Extract CDPP (Combined Differential Photometric Precision) from FITS header.
    CDPP measures photometric noise on transit timescales.
    Lower CDPP = better chance of detecting small planets.
    Returns CDPP in ppm, or None if not found.
    """
    fits_path = os.path.join(raw_dir, f'TIC_{tic_id}.fits')
    if not os.path.exists(fits_path):
        # Try labeled dir
        labeled_fits = glob.glob(f'../data/labeled/TIC_{tic_id}*.fits')
        if not labeled_fits:
            return None, None, None
        fits_path = labeled_fits[0]

    try:
        lc = lk.read(fits_path)
        # CDPP at different timescales (hours)
        cdpp_1h  = float(lc.estimate_cdpp(transit_duration=1  * u.hour).value)
        cdpp_2h  = float(lc.estimate_cdpp(transit_duration=2  * u.hour).value)
        cdpp_6h  = float(lc.estimate_cdpp(transit_duration=6  * u.hour).value)
        return cdpp_1h, cdpp_2h, cdpp_6h
    except:
        return None, None, None

print('CDPP function ready!')

CDPP function ready!


## 4. NEW — ACF Period Detection

In [4]:
def get_acf_period(time, flux, max_lag_days=13.0):
    """
    Compute Autocorrelation Function to find stellar rotation period.
    If ACF period matches transit period → likely stellar rotation, not planet.
    Returns ACF period in days, or 0 if not found.
    """
    try:
        dt       = np.median(np.diff(time))
        max_lag  = int(max_lag_days / dt)
        max_lag  = min(max_lag, len(flux) // 3)

        # Compute ACF
        flux_norm = (flux - np.mean(flux)) / np.std(flux)
        acf = np.correlate(flux_norm, flux_norm, mode='full')
        acf = acf[len(acf)//2:]
        acf = acf[:max_lag] / acf[0]

        lags_days = np.arange(len(acf)) * dt

        # Find first significant peak after lag=0
        min_lag_idx = int(0.5 / dt)   # Minimum 0.5 day lag
        peaks, props = find_peaks(
            acf[min_lag_idx:],
            height=0.1,       # Must be at least 10% of max
            distance=int(0.5/dt)
        )

        if len(peaks) == 0:
            return 0.0, 0.0

        best_peak     = peaks[np.argmax(props['peak_heights'])] + min_lag_idx
        acf_period    = float(lags_days[best_peak])
        acf_strength  = float(acf[best_peak])
        return acf_period, acf_strength

    except:
        return 0.0, 0.0

print('ACF function ready!')

ACF function ready!


## 5. NEW — Lomb-Scargle Period

In [5]:
def get_ls_period(time, flux, min_period=0.5, max_period=13.0):
    """
    Run Lomb-Scargle periodogram to find sinusoidal variability.
    If LS period matches transit period → stellar variability, not planet.
    Returns LS period, LS power.
    """
    try:
        frequency, power = LombScargle(time, flux).autopower(
            minimum_frequency = 1.0 / max_period,
            maximum_frequency = 1.0 / min_period
        )
        periods   = 1.0 / frequency
        best_idx  = np.argmax(power)
        ls_period = float(periods[best_idx])
        ls_power  = float(power[best_idx])

        # False alarm probability from LS
        ls_fap = float(LombScargle(time, flux).false_alarm_probability(
            power[best_idx], method='baluev'
        ))

        return ls_period, ls_power, ls_fap
    except:
        return 0.0, 0.0, 1.0

print('Lomb-Scargle function ready!')

Lomb-Scargle function ready!


## 6. Original Feature Functions (unchanged)

In [6]:
def phase_fold(time, flux, period, t0):
    phase = ((time - t0) % period) / period
    phase[phase > 0.5] -= 1.0
    sort_idx = np.argsort(phase)
    return phase[sort_idx], flux[sort_idx]

def get_transit_mask(phase, duration, period):
    half = (duration / period) / 2.0
    return np.abs(phase) < half

def odd_even_depth(time, flux, period, t0, duration):
    half = duration / 2.0
    transit_times = []
    n_transits = int((time[-1] - time[0]) / period)
    for i in range(n_transits + 1):
        tc   = t0 + i * period
        mask = np.abs(time - tc) < half
        if mask.sum() > 3:
            transit_times.append((i, 1.0 - np.median(flux[mask])))
    if len(transit_times) < 2:
        return 0.0, 0.0, 0.0
    odd_depths  = [d for i, d in transit_times if i % 2 == 1]
    even_depths = [d for i, d in transit_times if i % 2 == 0]
    odd_mean    = np.mean(odd_depths)  if odd_depths  else 0.0
    even_mean   = np.mean(even_depths) if even_depths else 0.0
    return odd_mean, even_mean, abs(odd_mean - even_mean)

def secondary_eclipse_depth(phase, flux, duration, period):
    sec_phase = phase.copy()
    sec_phase[sec_phase < 0] += 1.0
    half      = (duration / period) / 2.0
    sec_mask  = np.abs(sec_phase - 0.5) < half
    if sec_mask.sum() < 3:
        return 0.0
    out_mask  = (np.abs(phase) > half * 2) & (np.abs(sec_phase - 0.5) > half * 2)
    baseline  = np.median(flux[out_mask]) if out_mask.sum() > 3 else 1.0
    sec_depth = baseline - np.median(flux[sec_mask])
    return max(sec_depth, 0.0)

print('Feature helper functions ready!')

Feature helper functions ready!


## 7. Main Feature Extraction Function (UPGRADED)

In [7]:
def extract_features(tic_id, time, flux, bls_row):
    period   = bls_row['period_days']
    duration = bls_row['duration_days']
    depth    = bls_row['depth_ppm'] / 1e6
    t0       = bls_row['transit_time']
    snr      = bls_row['snr']

    phase, flux_folded = phase_fold(time, flux, period, t0)
    in_transit  = get_transit_mask(phase, duration, period)
    out_transit = ~in_transit

    feat = {
        'tic_id'         : tic_id,
        'period_days'    : period,
        'duration_days'  : duration,
        'duration_hours' : duration * 24,
        'depth_ppm'      : bls_row['depth_ppm'],
        'snr'            : snr,
        'bls_power'      : bls_row.get('bls_power', 0),
    }

    # ── TLS features (NEW) ───────────────────────────────────
    feat['tls_snr']       = float(bls_row.get('tls_snr', 0) or 0)
    feat['tls_sde']       = float(bls_row.get('tls_sde', 0) or 0)
    feat['tls_fap']       = float(bls_row.get('tls_fap', 1) or 1)
    feat['tls_rp_rs']     = float(bls_row.get('tls_rp_rs', 0) or 0)
    feat['tls_odd_even']  = float(bls_row.get('tls_odd_even', 0) or 0)
    feat['both_agree']    = int(bls_row.get('both_agree', 0) or 0)

    # ── Shape features ──────────────────────────────────────
    if in_transit.sum() > 3 and out_transit.sum() > 3:
        flux_in  = flux_folded[in_transit]
        flux_out = flux_folded[out_transit]
        feat['transit_depth_measured'] = float(np.median(flux_out) - np.median(flux_in))
        feat['flux_in_scatter']        = float(np.std(flux_in))
        feat['flux_out_scatter']       = float(np.std(flux_out))
        feat['in_out_scatter_ratio']   = float(np.std(flux_in) / (np.std(flux_out) + 1e-10))
        feat['transit_skewness']       = float(stats.skew(flux_in))     if len(flux_in)>3 else 0.0
        feat['transit_kurtosis']       = float(stats.kurtosis(flux_in)) if len(flux_in)>3 else 0.0
    else:
        for k in ['transit_depth_measured','flux_in_scatter','flux_out_scatter',
                  'in_out_scatter_ratio','transit_skewness','transit_kurtosis']:
            feat[k] = 0.0

    # ── Odd/Even & Secondary ────────────────────────────────
    odd_d, even_d, oe_diff = odd_even_depth(time, flux, period, t0, duration)
    feat['odd_depth']       = odd_d
    feat['even_depth']      = even_d
    feat['odd_even_diff']   = oe_diff
    feat['secondary_depth'] = secondary_eclipse_depth(phase, flux_folded, duration, period)

    # ── Statistical features ────────────────────────────────
    feat['flux_mean']       = float(np.mean(flux))
    feat['flux_std']        = float(np.std(flux))
    feat['flux_range']      = float(np.max(flux) - np.min(flux))
    feat['outlier_frac']    = float(np.sum(np.abs(flux-np.mean(flux)) > 3*np.std(flux)) / len(flux))
    feat['depth_over_std']  = float(depth / (np.std(flux) + 1e-10))
    feat['period_over_dur'] = float(period / (duration + 1e-10))

    # ── NEW: ACF period ─────────────────────────────────────
    acf_period, acf_strength = get_acf_period(time, flux)
    feat['acf_period']       = acf_period
    feat['acf_strength']     = acf_strength
    # Does ACF match transit period? → stellar rotation indicator
    feat['acf_period_match'] = int(
        acf_period > 0 and abs(acf_period - period) / period < 0.1
    )

    # ── NEW: Lomb-Scargle ───────────────────────────────────
    ls_period, ls_power, ls_fap = get_ls_period(time, flux)
    feat['ls_period']        = ls_period
    feat['ls_power']         = ls_power
    feat['ls_fap']           = ls_fap
    # Does LS match transit period? → sinusoidal variability indicator
    feat['ls_period_match']  = int(
        ls_period > 0 and abs(ls_period - period) / period < 0.1
    )

    return feat

print('Full feature extraction function ready!')

Full feature extraction function ready!


## 8. Extract Features for ALL Stars

In [8]:
all_features = []

for _, bls_row in tqdm(bls_df.iterrows(), total=len(bls_df), desc='Extracting features'):
    tic_id   = str(bls_row['tic_id']).replace('.0','')
    csv_path = os.path.join(PROCESSED_DIR, f'TIC_{tic_id}.csv')

    if not os.path.exists(csv_path):
        continue

    try:
        df   = pd.read_csv(csv_path)
        time = df['time'].values
        flux = df['flux'].values
        mask = np.isfinite(time) & np.isfinite(flux)
        time, flux = time[mask], flux[mask]

        if len(time) < 50:
            continue

        feat = extract_features(tic_id, time, flux, bls_row)

        # ── NEW: Add CDPP ─────────────────────────────────
        cdpp_1h, cdpp_2h, cdpp_6h = get_cdpp(tic_id, RAW_DIR)
        feat['cdpp_1h'] = cdpp_1h if cdpp_1h else np.std(flux) * 1e6
        feat['cdpp_2h'] = cdpp_2h if cdpp_2h else np.std(flux) * 1e6
        feat['cdpp_6h'] = cdpp_6h if cdpp_6h else np.std(flux) * 1e6

        all_features.append(feat)
        print(f'TIC {tic_id} ✅  '
              f'SNR={feat["snr"]:.1f}  '
              f'ACF={feat["acf_period"]:.2f}d  '
              f'LS={feat["ls_period"]:.2f}d  '
              f'CDPP_2h={feat["cdpp_2h"]:.0f}ppm')

    except Exception as e:
        print(f'TIC {tic_id} ❌ {e}')

features_df = pd.DataFrame(all_features)
features_df.to_csv(OUTPUT_FILE, index=False)
print(f'\nFeatures saved! Shape: {features_df.shape}')
print(f'Feature columns ({len(features_df.columns)}):')
for c in features_df.columns:
    print(f'  {c}')

Extracting features:   7%|▋         | 1/15 [00:02<00:33,  2.39s/it]

TIC 149603524 ✅  SNR=6.5  ACF=0.00d  LS=1.11d  CDPP_2h=879ppm


Extracting features:  13%|█▎        | 2/15 [00:05<00:34,  2.67s/it]

TIC 229742722 ✅  SNR=4.4  ACF=0.00d  LS=0.55d  CDPP_2h=1012ppm


Extracting features:  20%|██        | 3/15 [00:07<00:28,  2.39s/it]

...

TIC 261155555 ✅  SNR=35.6  ACF=0.68d  LS=0.57d  CDPP_2h=1675ppm
TIC 261203535 ❌ cannot convert float NaN to integer
TIC 271893367 ✅  SNR=11.3  ACF=0.52d  LS=5.41d  CDPP_2h=510ppm


Extracting features:  80%|████████  | 12/15 [00:22<00:04,  1.34s/it]

TIC 279741379 ✅  SNR=16.0  ACF=0.00d  LS=0.51d  CDPP_2h=380ppm


Extracting features: 100%|██████████| 15/15 [00:24<00:00,  1.61s/it]

TIC 350618622 ✅  SNR=3.8  ACF=0.00d  LS=0.51d  CDPP_2h=478ppm
TIC 441075486 ✅  SNR=9.2  ACF=4.40d  LS=2.26d  CDPP_2h=1142ppm
TIC 55525572 ✅  SNR=8.6  ACF=0.52d  LS=4.84d  CDPP_2h=324ppm

Features saved! Shape: (14, 39)
Feature columns (39):
  tic_id
  period_days
  duration_days
  duration_hours
  depth_ppm
  snr
  bls_power
  tls_snr
  tls_sde
  tls_fap
  tls_rp_rs
  tls_odd_even
  both_agree
  transit_depth_measured
  flux_in_scatter
  flux_out_scatter
  in_out_scatter_ratio
  transit_skewness
  transit_kurtosis
  odd_depth
  even_depth
  odd_even_diff
  secondary_depth
  flux_mean
  flux_std
  flux_range
  outlier_frac
  depth_over_std
  period_over_dur
  acf_period
  acf_strength
  acf_period_match
  ls_period
  ls_power
  ls_fap
  ls_period_match
  cdpp_1h
  cdpp_2h
  cdpp_6h


## 9. Also Extract for Labeled Dataset

In [9]:
LABELED_PROCESSED = '../data/labeled/processed/'
LABELS_CSV        = '../data/labeled/labels.csv'

if not os.path.exists(LABELS_CSV):
    print('No labeled dataset — skipping')
else:
    labels_df     = pd.read_csv(LABELS_CSV)
    labeled_feats = []

    for _, row in tqdm(labels_df.iterrows(), total=len(labels_df),
                       desc='Labeled features'):
        tic_id = str(row['tic_id']).replace('.0','')
        label  = row['label']
        pattern = glob.glob(os.path.join(LABELED_PROCESSED, f'*{tic_id}*.csv'))
        if not pattern:
            continue
        try:
            df   = pd.read_csv(pattern[0])
            time = df['time'].values
            flux = df['flux'].values
            mask = np.isfinite(time) & np.isfinite(flux)
            time, flux = time[mask], flux[mask]

            # Quick BLS for labeled data
            from astropy.timeseries import BoxLeastSquares
            bls    = BoxLeastSquares(time * u.day, flux)
            durs   = np.linspace(0.01, 0.3, 10) * u.day
            pgram  = bls.autopower(durs, minimum_period=1.0*u.day,
                                   maximum_period=13.0*u.day)
            bi     = np.argmax(pgram.power)
            bls_st = bls.compute_stats(pgram.period[bi], pgram.duration[bi],
                                       pgram.transit_time[bi])
            dep    = float(bls_st['depth'][0])
            dep_e  = float(bls_st['depth'][1])

            fake_bls = pd.Series({
                'period_days'  : float(pgram.period[bi].value),
                'duration_days': float(pgram.duration[bi].value),
                'depth_ppm'    : dep * 1e6,
                'snr'          : dep/dep_e if dep_e>0 else 0,
                'bls_power'    : float(pgram.power[bi]),
                'transit_time' : float(pgram.transit_time[bi].value),
                'tls_snr': 0, 'tls_sde': 0, 'tls_fap': 1,
                'tls_rp_rs': 0, 'tls_odd_even': 0, 'both_agree': 0
            })

            feat = extract_features(tic_id, time, flux, fake_bls)
            cdpp_1h, cdpp_2h, cdpp_6h = get_cdpp(tic_id, '../data/labeled/')
            feat['cdpp_1h'] = cdpp_1h or np.std(flux)*1e6
            feat['cdpp_2h'] = cdpp_2h or np.std(flux)*1e6
            feat['cdpp_6h'] = cdpp_6h or np.std(flux)*1e6
            feat['label']   = label
            labeled_feats.append(feat)
            print(f'TIC {tic_id} ({label}) ✅')

        except Exception as e:
            print(f'TIC {tic_id} ❌ {e}')

    labeled_feat_df = pd.DataFrame(labeled_feats)
    labeled_feat_df.to_csv(LABELED_FEAT, index=False)
    print(f'\nLabeled features saved: {labeled_feat_df.shape}')
    print(labeled_feat_df[['tic_id','label','snr','tls_fap','cdpp_2h']].to_string())

Labeled features:   7%|▋         | 1/15 [00:13<03:07, 13.36s/it]

TIC 261136679 (planet) ✅


Labeled features:  13%|█▎        | 2/15 [00:26<02:48, 12.98s/it]

TIC 238004786 (planet) ✅


Labeled features:  20%|██        | 3/15 [00:37<02:27, 12.32s/it]

...

TIC 219016883 (eclipsing_binary) ✅


Labeled features:  93%|█████████▎| 14/15 [02:44<00:14, 14.50s/it]

TIC 284925600 (eclipsing_binary) ✅


Labeled features: 100%|██████████| 15/15 [02:51<00:00, 11.40s/it]

TIC 120896927 (eclipsing_binary) ✅

Labeled features saved: (15, 40)
       tic_id             label       snr  tls_fap       cdpp_2h
0   261136679            planet  0.003537      1.0    124.513180
1   238004786            planet  0.008233      1.0   1426.563185
2   307210830            planet  0.019863      1.0    793.643589
3   394137592            planet  0.002497      1.0    266.267549
4   460205581            planet  0.021195      1.0   1165.659588
5    25155310            planet  0.038340      1.0   1342.908215
6    29857954            planet  0.003329      1.0    485.483050
7    55525572            planet  0.002986      1.0    324.476081
8   259377017            planet  0.023227      1.0   1289.257909
9   149603524            planet  0.011116      1.0    878.860730
10  229804573  eclipsing_binary  2.610784      1.0  32077.141139
11  142087638  eclipsing_binary  0.004637      1.0    493.772809
12  219016883  eclipsing_binary  0.028571      1.0   5850.007795
13  284925600  eclips

---
## ✅ Done!
**New features added:**
- `tls_snr`, `tls_sde`, `tls_fap`, `tls_rp_rs`, `tls_odd_even` — from TLS
- `acf_period`, `acf_strength`, `acf_period_match` — stellar rotation
- `ls_period`, `ls_power`, `ls_fap`, `ls_period_match` — sinusoidal variability
- `cdpp_1h`, `cdpp_2h`, `cdpp_6h` — photometric noise level
- `both_agree` — BLS and TLS agree on period

**Total features: ~30 vs original 15**

**Next → 05_model_training.ipynb**